# Bước 3: Tích hợp Dữ liệu (Data Integration)
**Người thực hiện:** Hà Xuân Khoa (Vị trí 0 - Nhóm trưởng)

Notebook này thực hiện quá trình tích hợp dữ liệu bao gồm:
1. **Vertical Concatenation:** Gộp các nguồn dữ liệu tĩnh (Tesco, Tiki) thành `Unified Static Master Table`.
2. **Horizontal Merging:** Kết nối lịch sử giao dịch giả lập (`Dynamic Data`) với `Unified Static Master Table`.


# 1. ĐỌC DỮ LIỆU VÀ IMPORT THƯ VIỆN


In [42]:
import pandas as pd
df_tesco = pd.read_csv('../data/processed/tesco_products_vnd.csv')
df_tiki = pd.read_csv('../data/processed/vietnamese_tiki_products_backpacks_suitcases.csv')
df_dynamic = pd.read_csv('../data/processed/dynamic_transactions.csv')

print("Tesco:", df_tesco.shape)
print("Tiki:", df_tiki.shape)
print("Dynamic:", df_dynamic.shape)

Tesco: (1200, 86)
Tiki: (5361, 18)
Dynamic: (10000, 12)


## 2. KIỂM TRA VÀ ĐỒNG BỘ CỘT

Hai nguồn Static có các thuộc tính sản phẩm chính:

- Product_ID
- Product_Name
- Category
- Brand
- Original_Price
- Discount_Price

Do dữ liệu Tiki có một số thông tin khác với Tesco,
bổ sung các cột cần thiết để hai nguồn có thể kết hợp bằng `pd.concat()`.

In [43]:
print("Tesco columns:", df_tesco.columns.tolist())
print("Tiki columns:", df_tiki.columns.tolist())
print("Dynamic columns:", df_dynamic.columns.tolist())

print("\nTesco dtypes:")
print(df_tesco.dtypes)

print("\nTiki dtypes:")
print(df_tiki.dtypes)

print("\nDynamic dtypes:")
print(df_dynamic.dtypes)

Tesco columns: ['Product_Name', 'Product_ID', 'tpnc', 'tpnb', 'gtin', 'gtin13', 'Discount_Price', 'unit_price', 'unit_of_measure', 'currency', 'availability', 'is_for_sale', 'base_product_id', 'shelfLife', 'foodIcons', 'restrictions', 'manufacturer', 'manufacturer_address', 'images', 'description', 'product_url', 'Brand', 'avg_rating', 'reviews_count', 'uniq_id', 'pack_size', 'alcohol_units', 'abv', 'Category', 'category_2', 'category_3', 'breadcrumbs', 'ingredients', 'nutrition', 'specifications', 'features', 'warnings', 'storage', 'other_information', 'allergens', 'preparation_and_usage', 'brand_marketing', 'product_marketing', 'origin_information', 'legal_labelling', 'nutritional_claims', 'directions', 'preparation_guidelines', 'cooking_instructions', 'clothing_info', 'product_dimensions', 'recycling_info', 'manufacturer_marketing', 'other_nutrition_information', 'pack_size_info', 'is_new', 'product_type', 'super_department', 'department', 'aisle', 'distributor_address', 'importer_a

## 3. CHUẨN HÓA PRODUCT_ID

Sử dụng `Product_ID` làm khóa liên kết giữa Static Data và Dynamic Data.

In [45]:
df_tesco['Product_ID'] = df_tesco['Product_ID'].astype(str)
df_tiki['Product_ID'] = df_tiki['Product_ID'].astype(str)
df_dynamic['Product_ID'] = df_dynamic['Product_ID'].astype(str)

## 4. BỔ SUNG CÁC CỘT CHO TIKI

Tiki sử dụng VND ngay từ nguồn dữ liệu. Bổ sung các cột tương ứng với Tesco để thống nhất thông tin về giá và đơn vị tiền tệ.

In [46]:

df_tiki["Source_Currency"] = "VND"

df_tiki["Original_Price_VND"] = df_tiki["Original_Price"]

df_tiki["Discount_Price_VND"] = df_tiki["Discount_Price"]

print("Tiki columns:", df_tiki.columns.tolist())

Tiki columns: ['Product_ID', 'Product_Name', 'description', 'Original_Price', 'Discount_Price', 'fulfillment_type', 'Brand', 'review_count', 'rating_average', 'favourite_count', 'pay_later', 'current_seller', 'date_created', 'number_of_images', 'vnd_cashback', 'has_video', 'Category', 'quantity_sold', 'Source_Currency', 'Original_Price_VND', 'Discount_Price_VND']


## 5. CHUẨN BỊ DỮ LIỆU STATIC

Hai nguồn Static có số lượng và loại thuộc tính khác nhau.

Giữ lại các thuộc tính hiện có của từng nguồn để không làm mất thông tin có giá trị cho các bước phân tích sau.

Thêm cột `Source` để xác định nguồn dữ liệu của từng sản phẩm.

Kiểm tra và loại bỏ các bản ghi `Product_ID` bị trùng trong dữ liệu Tiki trước khi thực hiện Integration để đảm bảo khóa sản phẩm duy nhất.

In [47]:
duplicate_tiki = df_tiki[df_tiki['Product_ID'].duplicated(keep=False)].sort_values('Product_ID')
print("Các Product_ID bị trùng:")
print(duplicate_tiki[['Product_ID', 'Product_Name', 'Original_Price_VND', 'Discount_Price_VND']])
print("\nSố dòng duplicate:", len(duplicate_tiki))

Các Product_ID bị trùng:
     Product_ID                       Product_Name  Original_Price_VND  \
4962  169624384         vali màu hồng nhạt 24 inch              380000   
5234  169624384         vali màu hồng nhạt 24 inch              380000   
4562  182797637  Áo Trùm Vải Không Dệt Vali SAKOS              239000   
5268  182797637  Áo Trùm Vải Không Dệt Vali SAKOS              239000   

      Discount_Price_VND  
4962              380000  
5234              380000  
4562              155000  
5268              155000  

Số dòng duplicate: 4


In [48]:
df_tiki = df_tiki.drop_duplicates(subset=['Product_ID'], keep='first').copy()

In [49]:

df_tesco_static = df_tesco.copy()
df_tiki_static = df_tiki.copy()

df_tesco_static['Source'] = "Tesco"
df_tiki_static['Source'] = "Tiki"

## 6. VERTICAL CONCATENATION

Gộp hai nguồn Static theo chiều dọc bằng `pd.concat()` để tạo `Unified Static Master Table`.


In [50]:
static_master = pd.concat([df_tesco_static, df_tiki_static], ignore_index=True)
print("Static master:", static_master.shape)
static_master.head(5)

Static master: (6559, 98)


,Product_Name,Product_ID,tpnc,tpnb,gtin,gtin13,Discount_Price,unit_price,unit_of_measure,currency,...,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,quantity_sold
0,Malibu Original Coconut Rum 1L,254870847,254870847.0,57420176.0,8.410025e+12,8.410025e+12,22.5,22.5,litre,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Walkers Wotsits Cheese Multipack Crisps 12x16.5g,261495181,261495181.0,60266089.0,5.000328e+12,5.000328e+12,2.99,15.1,kg,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Listerine Total Care Intense Taste Classic Mou...,262720979,262720979.0,61463009.0,5.010124e+12,5.010124e+12,5.25,10.5,litre,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Walkers Roast Chicken Multipack Crisps 6 x 25g,321658606,321658606.0,96874748.0,5.000328e+12,5.000328e+12,2.2,14.67,kg,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Walkers Smokey Bacon Multipack Crisps 6 x 25g,321769965,321769965.0,96884385.0,5.000328e+12,5.000328e+12,2.2,14.67,kg,GBP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. KIỂM TRA STATIC MASTER

Kiểm tra số lượng sản phẩm và `Product_ID` duy nhất sau khi gộp.

In [51]:
print("Tổng số dòng:", len(static_master))
print("Product_ID unique:", static_master['Product_ID'].nunique())

if len(static_master) == static_master["Product_ID"].nunique():
    print("Product_ID không bị trùng.")
else:
    print("Vẫn còn Product_ID bị trùng.")

Tổng số dòng: 6559
Product_ID unique: 6559
Product_ID không bị trùng.


## 8. CHUẨN BỊ STATIC DATA CHO MERGE

Chọn các thuộc tính Static có giá trị đối với dữ liệu giao dịch và phân tích.

`Product_ID` được sử dụng làm khóa liên kết với Dynamic Data.

In [53]:
static_columns = [
    'Product_ID',
    'Product_Name',
    'Category',
    'Brand',
    'Original_Price',
    'Discount_Price',
    'Original_Price_VND',
    'Discount_Price_VND',
    'Source_Currency',
    'avg_rating',
    'reviews_count',
    'rating_average',
    'review_count',
    'quantity_sold',
    'availability',
    'is_for_sale',
    'description',
    'product_type',
    'category_2',
    'category_3',
    'manufacturer',
    'current_seller',
    'fulfillment_type',
    'date_created',
    'number_of_images',
    'favourite_count',
    'has_video',
    'Source'
]
static_features = static_master[[col for col in static_columns if col in static_master.columns]].copy()
print("Số cột Static được đưa vào Merge:", len(static_features.columns))

Số cột Static được đưa vào Merge: 28


## 9. HORIZONTAL MERGING

Kết nối `Dynamic Data` với `Unified Static Master Table` thông qua `Product_ID`.

Sử dụng `LEFT JOIN` để giữ lại toàn bộ lịch sử giao dịch.

In [54]:
master_dataset = pd.merge(df_dynamic, static_features, on = 'Product_ID', how='left')
print("Dynamic shape:", df_dynamic.shape)
print("Master Dataset shape:", master_dataset.shape)

Dynamic shape: (10000, 12)
Master Dataset shape: (10000, 39)


## 10. SẮP XẾP CÁC CỘT QUAN TRỌNG

Đưa các thuộc tính chính của giao dịch và sản phẩm lên đầu Dataset.

Các thuộc tính Static còn lại vẫn được giữ nguyên phía sau.

In [56]:
cols_order = [
    "Transaction_ID",
    "Transaction_Date",
    "Customer_ID",
    "Gender",
    "Age",
    "City",

    "Product_ID",
    "Product_Name",
    "Category",
    "Brand",

    "Quantity",
    "Unit_Price_VND",
    "Revenue",
    "Source_Currency",

    "Rating",
    "Customer_Review"
]

remaining_cols = [
    col for col in master_dataset.columns
    if col not in cols_order
]

master_dataset = master_dataset[
    cols_order + remaining_cols
]

print("Tổng số cột Master Dataset:", len(master_dataset.columns))

master_dataset.head()

Tổng số cột Master Dataset: 39


,Transaction_ID,Transaction_Date,Customer_ID,Gender,Age,City,Product_ID,Product_Name,Category,Brand,...,category_2,category_3,manufacturer,current_seller,fulfillment_type,date_created,number_of_images,favourite_count,has_video,Source
0,T000001,2026-05-27 07:53:23,C01049,Male,31,TP. Hồ Chí Minh,173798774,Hũ chiết kem dưỡng cao cấp nút nhân Rabbit Lab...,Túi chống sốc,OEM,...,NaN,NaN,NaN,Rabbit Lab,dropship,525.0,3.0,0.0,False,Tiki
1,T000002,2025-11-03 05:40:50,C00099,Female,47,Vũng Tàu,24659536,Bộ Chiết Mỹ Phẩm Du Lịch,Thời Trang,OEM,...,NaN,NaN,NaN,Hitech07,dropship,1504.0,7.0,0.0,False,Tiki
2,T000003,2025-09-30 17:07:01,C00503,Male,23,Huế,16683725,Balo túi du lịch đa năng Ozuko Z9060 (Đen),Thời Trang,OZUKO,...,NaN,NaN,NaN,Tỷ Lợi,dropship,1572.0,3.0,0.0,False,Tiki
3,T000004,2026-07-05 23:45:05,C00810,Male,59,Hà Nội,570490,Túi Kéo Du Lịch Macat Innova 4 - Đỏ,Thời Trang,Macat,...,NaN,NaN,NaN,MACAT OFFICIAL,dropship,0.0,5.0,0.0,False,Tiki
4,T000005,2026-01-05 19:27:32,C00241,Female,59,TP. Hồ Chí Minh,325087836,Nicola Spring Coir Door Mat - 60 x 40cm - Blac...,Marketplace,Nicola Spring,...,Home Accessories,Rugs & Mats,\N,NaN,NaN,NaN,NaN,NaN,NaN,Tesco


## 11. KIỂM TRA KẾT QUẢ MERGE

Kiểm tra số lượng dòng trước và sau khi Merge để đảm bảo không làm thay đổi số lượng giao dịch.

In [57]:
print("Số giao dịch ban đầu:", len(df_dynamic))
print("Số dòng sau merge:", len(master_dataset))
if len(df_dynamic) == len(master_dataset):
    print("Số lượng giao dịch được giữ nguyên")
else:
    print("Số dòng đã thay đổi, cần kiểm tra Product_ID")
print("\nMissing Values:")
print(master_dataset.isna().sum())

Số giao dịch ban đầu: 10000
Số dòng sau merge: 10000
Số lượng giao dịch được giữ nguyên

Missing Values:
Transaction_ID           0
Transaction_Date         0
Customer_ID              0
Gender                   0
Age                      0
City                     0
Product_ID               0
Product_Name             0
Category                 0
Brand                    0
Quantity                 0
Unit_Price_VND           0
Revenue                  0
Source_Currency          0
Rating                   0
Customer_Review          0
Original_Price           0
Discount_Price           0
Original_Price_VND       0
Discount_Price_VND       0
avg_rating            8178
reviews_count         8178
rating_average        1822
review_count          1822
quantity_sold         1822
availability          8178
is_for_sale           8178
description              0
product_type          8178
category_2            8178
category_3            8178
manufacturer          8178
current_seller        1822
fulf

## 12. LƯU MASTER DATASET

Lưu Dataset sau khi hoàn thành quá trình Integration vào thư mục `data/final`.

In [59]:
output_path = "../data/final/master_dataset.csv"

master_dataset.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Đã lưu Master Dataset: {output_path}")

Đã lưu Master Dataset: ../data/final/master_dataset.csv
